***The work below is based on Python for Genomic Data Science Course by John Hopkins. The first section is the preparation for the exam that is in the second section.***

Write a Python program that takes as input a file containing DNA sequences in multi-FASTA format, and computes the answers to the following questions. You can choose to write one program with multiple functions to answer these questions, or you can write several programs to address them. We will provide a multi-FASTA file for you, and you will run your program to answer the exam questions. 

While developing your program(s), please use the following example file to test your work: 
dna.example.fasta

You'll be given a different input file to launch the exam itself.

Here are the questions your program needs to answer. The quiz itself contains the specific multiple-choice questions you need to answer for the file you will be provided.

(1) How many records are in the file? A record in a FASTA file is defined as a single-line header, followed by lines of sequence data. The header line is distinguished from the sequence data by a greater-than (">") symbol in the first column. The word following the ">" symbol is the identifier of the sequence, and the rest of the line is an optional description of the entry. There should be no space between the ">" and the first letter of the identifier. 

(2) What are the lengths of the sequences in the file? What is the longest sequence and what is the shortest sequence? Is there more than one longest or shortest sequence? What are their identifiers? 

(3) In molecular biology, a reading frame is a way of dividing the DNA sequence of nucleotides into a set of consecutive, non-overlapping triplets (or codons). Depending on where we start, there are six possible reading frames: three in the forward (5' to 3') direction and three in the reverse (3' to 5'). For instance, the three possible forward reading frames for the sequence AGGTGACACCGCAAGCCTTATATTAGC are: 

AGG TGA CAC CGC AAG CCT TAT ATT AGC

A GGT GAC ACC GCA AGC CTT ATA TTA GC

AG GTG ACA CCG CAA GCC TTA TAT TAG C 

These are called reading frames 1, 2, and 3 respectively. An open reading frame (ORF) is the part of a reading frame that has the potential to encode a protein. It starts with a start codon (ATG), and ends with a stop codon (TAA, TAG or TGA). For instance, ATGAAATAG is an ORF of length 9.

Given an input reading frame on the forward strand (1, 2, or 3) your program should be able to identify all ORFs present in each sequence of the FASTA file, and answer the following questions: what is the length of the longest ORF in the file? What is the identifier of the sequence containing the longest ORF? For a given sequence identifier, what is the longest ORF contained in the sequence represented by that identifier? What is the starting position of the longest ORF in the sequence that contains it? The position should indicate the character number in the sequence. For instance, the following ORF in reading frame 1:

\>sequence1

ATGCCCTAG

starts at position 1.

Note that because the following sequence:

\>sequence2

ATGAAAAAA

does not have any stop codon in reading frame 1, we do not consider it to be an ORF in reading frame 1. 

(4) A repeat is a substring of a DNA sequence that occurs in multiple copies (more than one) somewhere in the sequence. Although repeats can occur on both the forward and reverse strands of the DNA sequence, we will only consider repeats on the forward strand here. Also we will allow repeats to overlap themselves. For example, the sequence ACACA contains two copies of the sequence ACA - once at position 1 (index 0 in Python), and once at position 3. Given a length n, your program should be able to identify all repeats of length n in all sequences in the FASTA file. Your program should also determine how many times each repeat occurs in the file, and which is the most frequent repeat of a given length.

In [125]:
from pprint import pprint

In [126]:
def extract_dna_seqs(file_name):
    '''Save the text in a FASTA file into a string.

        Args:
            file_name (str): The name of the FASTA file.

        Returns:
            string: The contents of the FASTA file in string format.
    
        Raises:
            FileNotFoundError: Only a valid file name in the current
            directory works. 
    '''
    try:
        with open(file_name, 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("File was not found or the file name is misspelled.")
        return None
    

def create_dna_dict(file_name):
    '''Create dictionary where keys are sequence IDs and their values are the sequences.

        Args:
            file_name (str): The name of the FASTA file.

        Returns:
            dict
    '''
    dna_seqs = extract_dna_seqs(file_name)
    dna_seq_dict = {}
    for line in dna_seqs.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            identifier = line[1:]
            dna_seq_dict[identifier] = ''
            continue

        dna_seq_dict[identifier] += line

    return dna_seq_dict

print(type(extract_dna_seqs("dna.example.fasta")))
dna_seqs = create_dna_dict("dna.example.fasta")
pprint(dna_seqs)
len(dna_seqs)


<class 'str'>
{'gi|142022655|gb|EQ086233.1|101 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': 'CACATCGACACGAAGATCACCGCGCATGCGTTGCTGATCACGCTCGTCAGCGCGCGTGCCTCGGACATGAAGCGATCGATGCCGACCAGCAGTGCGACACCGGCGACGGGCAGGTCGGGCATGACGACGAGCGTGGCGACCAGCGCAACCAGCCCGCTTCCGGAAACGCCGGCCGCGCCCTTGGACGTGAGCAGCATGATGGCGAGCATCACGGCGATCTGCGACGCGGAAAGGGGCACGTCGCACGCCTGCGCGATGAACAACGCGGCGAGCGTCAGATAGATCGCGGTACCGTCCAGATTGAACGAATAACCCGCCGGCAGCACGAGCCCCACGACGCCCTTGTCGCACCCGAGCGATTCCAGCTTGACGATCAGGCGTGGCAGAACGGGCTCCGAAGAGGACGTCGCGAGGACGATGAGCAACTCTTCGCGCAGGTAGCGCAAGAGCCGCCACAGCGCGAAGCCGTGCAGCCGCGCGAGCGGGGCGAGCACCAGTGCGACGAACAGCCCGCAGGCCACGTAGAAGGACAGCATCAGCTTCGCGAGCGAGCCGATCGAGCCGATTCCGAAGCGGCCCACCGTGAAGGCCATCGCGCCGAATGCGCCGAGCGGCGCGAGCCGCATGATCATCGCGAGCACGCGAAAGACGACCTGGGCGACGCCGTCGATCAGTGCAAGAACGGGCCGCCCGGCCCGCGGGTGTGCGTTCAGCGAGAAGCCGAACAACAGCGACAGCAGCAGCACCGGCAACACCTCGCCTTTCTCGAACGCGCCGAGCATCGTATCGGGGATCACGCTCAGCCCGAACGCGACGAGCCCGTTCGGTTGCGCGTCCCTCACGTACGGCGCGAGGATGCG

25

In [127]:
def number_of_seqs(file_name):
    '''Return a count of the number of sequences in a FASTA file.

    Args:
        file_name (str): Path of the FASTA file.

    Returns:
        int: Number of sequences in the file.
    '''
    dna_seqs = extract_dna_seqs(file_name)
    count = 0
    for line in dna_seqs:
        if line.startswith(">"):
            count += 1
            
    return count

count = number_of_seqs("dna.example.fasta")
print(count)



25


(2) What are the lengths of the sequences in the file? What is the longest sequence and what is the shortest sequence? Is there more than one longest or shortest sequence? What are their identifiers? 

In [128]:
def length_seqs_stats(file_name, output="general", print_output = False):
    '''Provide some simple statistics of the lengths of the sequences in a FASTA file.
    
    Args:
        file_name (str): Path of the FASTA file.

        output (str): Alters the return value of the function and its possible print
        into console. "general" by default. One of:
            - "general": This returns a dictionary of sequence lengths along with their 
                        identifiers, a dictionary of the longest sequence(s) along with
                        their identifiers, the length of the longest sequence, a
                        similar dictionary for the shortest sequence(s), and the length
                        of the shortest sequence.
            
            - "lengths": This returns a dictionary of sequence lengths along with their 
                        identifiers.

        print_output (bool): Boolean value that decides if function prints into console. False by
        default.
    
    Returns:
        tuple or dict:
            - If output == "general": a 5-item tuple of
            (lengths_id_dict, longest_seqs, longest_seq_len,
            shortest_seqs, shortest_seq_len).
            - If output == "lengths": a dict mapping sequence id to
            sequence length.
    
    Raises:
        ValueError: If unspecified value for output is inputted.
    '''
    dna_seqs_dict = create_dna_dict(file_name)

    lengths_id_dict = {seq_id:len(seq) for seq_id,seq in dna_seqs_dict.items()}

    longest_seq_len = max(lengths_id_dict.values())
    shortest_seq_len = min(lengths_id_dict.values())
    longest_seqs = {seq_id:seq for seq_id,seq in dna_seqs_dict.items() if len(seq) == longest_seq_len}
    shortest_seqs = {seq_id:seq for seq_id,seq in dna_seqs_dict.items() if len(seq) == shortest_seq_len}

    if output == "general":
        if print_output:
            print("General statistics of the provided DNA sequences:\n")
            print(f"Longest sequence(s):      Length: {longest_seq_len}")
            for seq_id, seq in longest_seqs.items():
                print(seq_id)
                print(seq)
                print()

            print()
            print(f"Shortest sequence(s):     Length: {shortest_seq_len}")
            for seq_id, seq in shortest_seqs.items():
                print(seq_id)
                print(seq)
                print()

        return_value = lengths_id_dict, longest_seqs, longest_seq_len, shortest_seqs, shortest_seq_len
        
    elif output == "lengths":     
        if print_output:
            print("Lengths of sequences:")
            for seq_id, length in lengths_id_dict.items():
                print(seq_id)
                print(length)

        return_value = lengths_id_dict

    else:
        raise ValueError(f"Unknown output type: {output}")

    return return_value
        
result = length_seqs_stats(file_name="dna.example.fasta", print_output=True)       

General statistics of the provided DNA sequences:

Longest sequence(s):      Length: 4805
gi|142022655|gb|EQ086233.1|323 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence
ACGCCCGGCGCACCGCGAGTACCGCGCCGCCGGGCACTCCTTGACCCCGCATGATCGATTCCCGATGAAACCCGAAAACCTCGTCGCCTGCCACGAATGCGACCTGCTGTTTTGGCGGCCGCCGCGCTTGCGCGCGCTGGCTGCGCACTGCCCGAGGTGCCGTGCCCGCGTGGGCGGCAGCGCGCACGGCCGTCCGGCGCTCGACCGGCGGTGCGCGATCGCGCTCGCCGCGCTGTTCACGCTCTTCATCGCGCAGGCCTTTCCCATCGTCGCGCTCGACGCCGCCGGCATCGCATCGCACGCGACGCTGGCCGACGCGGTGGCCGCGTTGCGCTTGAACGGGCAACCGGCGGTGGCGGCGATCGTGTTCTGCACGACGATGTTGTTCCCGCTGCTGGAACTCGCCGCGTGGCTGTACGTGCTCGTACCGTTGCGCGCGGGCCGCGTACCGCCCCGCTTCGAGCCGGTCCTGCGCAACATGCAGCGGCTGCGCCCGTGGAGCATGGTCGAGGTGTTCCTGCTCGGCATCCTGGTCACGATCGTCAAGATGACGAGCCTCGCGCACGTGATACCGGGCCCCGCGCTGTTTGCGTTCGGCGCCCTCACCGTGTTGCTCGGCTTTCTCGCGTCATTCGACCCGGGCGGCCTGTGGGAAGCGCGCGACGAAATCATCGCGCTGCGCGGCGGCGGTACGTCCGCCGCGGTATCGCGCCGGCGGCACACGCCGCGACGCGCTGCACCGGTGACGCCCGACACAGCGGACGCAACGAACGCGACCGGCGCGACCGG

(3) In molecular biology, a reading frame is a way of dividing the DNA sequence of nucleotides into a set of consecutive, non-overlapping triplets (or codons). Depending on where we start, there are six possible reading frames: three in the forward (5' to 3') direction and three in the reverse (3' to 5'). For instance, the three possible forward reading frames for the sequence AGGTGACACCGCAAGCCTTATATTAGC are: 

AGG TGA CAC CGC AAG CCT TAT ATT AGC

A GGT GAC ACC GCA AGC CTT ATA TTA GC

AG GTG ACA CCG CAA GCC TTA TAT TAG C 

These are called reading frames 1, 2, and 3 respectively. An open reading frame (ORF) is the part of a reading frame that has the potential to encode a protein. It starts with a start codon (ATG), and ends with a stop codon (TAA, TAG or TGA). For instance, ATGAAATAG is an ORF of length 9.

Given an input reading frame on the forward strand (1, 2, or 3) your program should be able to identify all ORFs present in each sequence of the FASTA file, and answer the following questions: what is the length of the longest ORF in the file? What is the identifier of the sequence containing the longest ORF? For a given sequence identifier, what is the longest ORF contained in the sequence represented by that identifier? What is the starting position of the longest ORF in the sequence that contains it? The position should indicate the character number in the sequence. For instance, the following ORF in reading frame 1:

\>sequence1

ATGCCCTAG

starts at position 1.

Note that because the following sequence:

\>sequence2

ATGAAAAAA

does not have any stop codon in reading frame 1, we do not consider it to be an ORF in reading frame 1.

In [129]:
def seq_rf(seq, reading_frame_pos=1):
    '''Convert a sequence into codons given a certain reading frame

    Args:
        seq (Str): A DNA sequence in string format.

        reading_frame_pos (int): A parameter representing reading frame 1, 2 or 3.

    Returns:
        list: A list of the reading frame.

    Raises:
        ValueError: reading_frame_pos can only take values 1, 2, or 3.
    '''
    if reading_frame_pos not in (1, 2, 3):
        raise ValueError("Incompatible value inputted for reading_frame parameter")

    i = reading_frame_pos - 1
    if reading_frame_pos > 1:
        codons = [seq[:i]]
    else:
        codons = []

    while i + 3 <= len(seq):
        codons.append(seq[i:i + 3])
        i += 3

    if i != len(seq):
        codons.append(seq[i:])

    return codons

def max_length_orf_total(seq_dict):
    '''Return longest orf in the entire file.
        Args:
            seq_dict (dict) - nested dictionary of sequences with their
            longest orf(s) and their respective positions.
        
        Returns:
            dict - A nested dictionary of all the longest ORFs with their
            positions in the sequence and the sequence ID.
    '''
    longest_len_so_far = 0
    longest_orf_so_far = {}

    for seq_id, orf_dict in seq_dict.items():
        orf_curr = next(iter(orf_dict.values()), None)
        if orf_curr is None:
            continue

        len_orf_curr = len(orf_curr)

        if len_orf_curr > longest_len_so_far:
            longest_len_so_far = len_orf_curr
            longest_orf_so_far = {seq_id: orf_dict}
        elif len_orf_curr == longest_len_so_far:
            longest_orf_so_far.update({seq_id: orf_dict})

    return longest_orf_so_far

def orf(file_name, reading_frame_pos=1):
    '''Return a dictionary of all open reading frames (ORFs), longest ORFs
    per sequence, and the longest ORF in the FASTA file.

        Args:
            file_name (str): Path of the FASTA file.

            reading_frame_pos (int): A parameter representing reading
            frame 1, 2 or 3.

        Returns:
            dict:
                - Return three dictionaries containing all ORFs, longest ORF(s)
                per sequence and the longest ORF(s).
    '''

    # Produce a nested dictionary with structure: {seq_id: {orf_pos: orf}}
    dna_seq_dict = create_dna_dict(file_name)
    seq_orfs_dict = {}
    rf_pos_list = [1, -1, 0]
    rf_pos = rf_pos_list[reading_frame_pos - 1]

    for seq_id, seq in dna_seq_dict.items():
        reading_frame = seq_rf(seq, reading_frame_pos)

        start_codon_indices = [
            index
            for index, value in enumerate(reading_frame)
            if value == "ATG"
        ]
        stop_codon_indices = [
            index
            for index, value in enumerate(reading_frame)
            if value in ("TAA", "TAG", "TGA")
        ]

        prev_stop_codon = 0
        orfs_for_curr_seq = {}

        for stop_codon_index in stop_codon_indices:
            # This finds all start codons before the current
            # stop codon after the previous stop codon allowing
            # me to isolate the possible ORF(s).
            start_codon_indices_before = [
                start_codon_index
                for start_codon_index in start_codon_indices
                if prev_stop_codon <= start_codon_index < stop_codon_index
            ]

            for scib in start_codon_indices_before:
                orfs_for_curr_seq[f"{scib * 3 + rf_pos}-{stop_codon_index * 3 + rf_pos + 2}"] = "".join(
                    reading_frame[scib:stop_codon_index + 1]
                )

            prev_stop_codon = stop_codon_index

        seq_orfs_dict[seq_id] = orfs_for_curr_seq

    # Produce a nested dictionary that has structure: {seq_id: {seq_pos: longest_orf_for_given_seq}}.
    longest_orf_per_seq = {
        seq_id:{
            orf_pos:longest_orf_per_sequence
            for orf_pos, longest_orf_per_sequence in orfs_dict.items()
            if  len(longest_orf_per_sequence) == len(max(orfs_dict.values(), key=len))
        }
        for seq_id, orfs_dict in seq_orfs_dict.items()
    }

    # Produce a nested dictionary that has structure: {seq_id: {seq_pos: longest_orf}}.
    longest_orf = max_length_orf_total(longest_orf_per_seq)

    return seq_orfs_dict, longest_orf_per_seq, longest_orf

seq_orfs_dict, longest_orf_per_seq, longest_orf = orf("dna.example.fasta")
# print("All ORFs for each sequence:")
# pprint(seq_orfs_dict)
# print("Longest ORF for each sequence:")
# pprint(longest_orf_per_seq)
print("Longest orf in dna.example.fasta:")
pprint(longest_orf)

Longest orf in dna.example.fasta:
{'gi|142022655|gb|EQ086233.1|323 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'2824-4509': 'ATGCGAACCAGTTGTTCGGACACCTCGACGAGCAAGTCGTGCCGCAGGCGCGCGACACGCTGGCGGCGGCGCAACGCACGTTCGACGCCGCGCAGGCGACGCTGCGGCAGGATTCGCCGATGCAATCGGACGTTCATGACGCGATGCAATCGCTCACGCAGACGCTCCAGTCGCTCAATACGCTGGCCGACTATCTCGAGCGGCATCCGGAAGCGCTGCTCTTCGGCAAGAAAGGAGAACCGAAATGACGCCGCATTCGTTTCGCACGTTCCGGATGCCGATCCGCATCGCGACGTGCATCGCGCTGGCCGTGCTCGGCGCCTGTACGTCGCCGCCCGTACGGTTCCATACGCTCGGGATGGCGGATGGCGCGGGCGGCGACACCGACGCCTCGCGTCCCGCATGGCTGATCGACATGCAGCGCGTGCACGTGGCGGCGCCGGCGGACGGCAACCGGCTCGCGGTGCAGCGCGGCCCCGAACGGGTCGACATCCTGGAACAGGAGCGCTGGGTTGCGCCGCTCGGCGACGAGATGCGCGACGGACTGTCGACGCGCGTCACGTCCCGGCTGAACACGTTCGACGTTCACCGCGTCGCTCATCCGGATGGCACGCCGGTCTATCGGGTCGCCGTGGACGTCCAGCGTGTCGAATCGTGGCCGGCGTCTCACGTGCTGCTCGATGCGACGTGGACGGTGGACGCCGGCTCAGGACAGCCGGCACTGACTTGCCGCAGCATCGTTCGGGCCGGTGCGTCGGCGGGCTACGACGCGCTCGTCGACGCGCATCGCCATGCGCTCGACACGCTCGCGCTCGGCATCGCCGCC

---
# **Important Issues**

### Storing FASTA as string is not efficient

The main inefficiency is at the very beginning. The storing of the entire file as a string takes up a lot of memory. This is not efficient and can lead to overflow

### Dictionaries are inefficient in our case

The code at its current state is not efficient. While this may be okay for our example and the exam this code will be used for, the real use case may contain files in the *gigabytes* of size. The dictionary and nested dictionary utilised here will break at that point as RAM overflow may become an issue. This adds upon the previous issue meaning peak memory usage would be the addition of both so this is a layered problem.

### Solution

After writing my code, I asked claude to suggest more efficient structures for my use case and it suggested creating a new dataclass structure within python and generators whereby the entire file is not stored in memory and instead we will go sequence by sequence which allows the code to run in memory proportional to the size of the individual sequence.

---

(4) A repeat is a substring of a DNA sequence that occurs in multiple copies (more than one) somewhere in the sequence. Although repeats can occur on both the forward and reverse strands of the DNA sequence, we will only consider repeats on the forward strand here. Also we will allow repeats to overlap themselves. For example, the sequence ACACA contains two copies of the sequence ACA - once at position 1 (index 0 in Python), and once at position 3. Given a length n, your program should be able to identify all repeats of length n in all sequences in the FASTA file. Your program should also determine how many times each repeat occurs in the file, and which is the most frequent repeat of a given length.

In [130]:
# I will first implement a version of the code that does
# not include the improvements.
def repeats_inefficient(file_name: str, length: int):
    """Return amount of all repeats of a certain length.

    Args:
        file_name (str): The name of the FASTA file.
        length (int): The length of the repeats that are considered.

    Returns:
        dict: A dictionary with the repeats as the keys and their
            frequencies as the values.
    """
    dna_dict = create_dna_dict(file_name)
    repeats_dict = {}

    for seq in dna_dict.values():
        seq_pos = 0

        while seq_pos + length <= len(seq):
            sub_string = seq[seq_pos: seq_pos + length]

            if repeats_dict.get(sub_string) is not None:
                repeats_dict[sub_string] += 1
            else:
                repeats_dict[sub_string] = 1

            seq_pos += 1

    return repeats_dict

repeats_dict = repeats_inefficient('dna.example.fasta', length=3)
pprint(repeats_dict)
most_freq = max(repeats_dict, key=repeats_dict.get)
print(f"The most frequent repeat in the file is {most_freq}" +
      f" with {repeats_dict[most_freq]} instances.")

{'AAA': 393,
 'AAC': 593,
 'AAG': 546,
 'AAT': 419,
 'ACA': 506,
 'ACC': 725,
 'ACG': 1421,
 'ACT': 287,
 'AGA': 401,
 'AGC': 1194,
 'AGG': 565,
 'AGT': 334,
 'ATA': 263,
 'ATC': 1084,
 'ATG': 778,
 'ATT': 442,
 'CAA': 546,
 'CAC': 915,
 'CAG': 993,
 'CAT': 864,
 'CCA': 746,
 'CCC': 722,
 'CCG': 2019,
 'CCT': 540,
 'CGA': 2054,
 'CGC': 2810,
 'CGG': 2111,
 'CGT': 1369,
 'CTA': 136,
 'CTC': 707,
 'CTG': 903,
 'CTT': 514,
 'GAA': 873,
 'GAC': 1106,
 'GAG': 765,
 'GAT': 1056,
 'GCA': 1393,
 'GCC': 1894,
 'GCG': 2920,
 'GCT': 1035,
 'GGA': 706,
 'GGC': 1893,
 'GGG': 748,
 'GGT': 782,
 'GTA': 389,
 'GTC': 1059,
 'GTG': 891,
 'GTT': 599,
 'TAA': 139,
 'TAC': 324,
 'TAG': 191,
 'TAT': 228,
 'TCA': 670,
 'TCC': 686,
 'TCG': 1987,
 'TCT': 397,
 'TGA': 637,
 'TGC': 1343,
 'TGG': 704,
 'TGT': 454,
 'TTA': 94,
 'TTC': 894,
 'TTG': 567,
 'TTT': 408}
The most frequent repeat in the file is GCG with 2920 instances.


# Real Exam Questions



***1. How many records are there in the `dna2.fasta` file?***

In [131]:
print(number_of_seqs('dna2.fasta'))

18


***2. What is the length of the longest sequence? 3. What is the length of the shortest sequence?***

In [132]:
result_value = length_seqs_stats(file_name="dna2.fasta", print_output=True)

General statistics of the provided DNA sequences:

Longest sequence(s):      Length: 4894
gi|142022655|gb|EQ086233.1|255 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence
CTCGACGCGCTCCGCGTCGAGGTCGCCCGACGTCTCGCGCAGCAACTGATTCAAAAACAGGCCGCCGCTCATGCCGATCTTGCGGTGGATGCGCCACACCGACAGTTCGATGCCTTCGGCATCGAGCGCTTCCTTCCACGCAAGCACGTGCTGGTAGACGCTGTCGACGAGCGTGCCGTCGAGATCGAACAGAAAAGACGTTTCAATGCGCATGTGTATCTCCTGGCTCGAAAGGGGGCGAGCGAACGGGTCGTGAAGCGTGTCCGCACATTATCGGCGCGCGGCGATGTCATGACCATGTCCCGCGGCCCGCCGACGCGACGCCACCCCGTGCCGCGCCGTGTCATGTGCCGCTGGTACAATCGCGGCGATCGCCGGGCCGGGCTCTCCGCGCCGCGCGCCCCCAACCCTCGTCTCGCCGATTCCAGGTATGGCTACACCGGACGCCGTCAGTTCCAAGCACTCGTGGTGGGGTGTCCTGGCCCTGGCACTCACCGCCTTCATCTTCAATACCACCGAATTCGTGCCCGTCGCGCTGCTCAGCGCGATCGGCGACAGCCTGCACATGCAGCCGACCGACGTCGGCCTGATGCTGACGATCTACGCGTGGGCCGTGGCCGTCGTGTCCTTGCCGCTGACGCTGGCCACGCGCCACGTCGAGCGCCGCAAGCTGCTGACGGGGGCATTGCTGGTATTCATCGCGAGCCACGTCGTGACCGGTGTCGCGTGGAATTTCGCGGTGCTGATGGTCGGCCGGCTGGGCATCGCATGTGCGCATGCGGTGTTCTG

***4. What is the length of the longest ORF appearing in reading frame 2 of any of the sequences?***

In [133]:
rfp = 2
orf_length_dict3, orf_longest_seq_dict3, orf_longest_file3 = orf("dna2.fasta", reading_frame_pos=rfp)

pprint(orf_longest_file3)
inner_dict = next(iter(orf_longest_file3.values()))
print(f"The length of the longest ORF in reading frame {rfp} in the whole file is {len(next(iter(inner_dict.values())))}")
print()
# pprint(orf_longest_seq_dict)

{'gi|142022655|gb|EQ086233.1|16 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'3071-4528': 'ATGGCAATCCTGATTCGTGGCGGCACCGTGGTCGATGCGGACCGTTCCTACCGCGCGGACGTGCTCTGCGCAGCCCCGGAGGACGGCGGCACGATCCTGCAGATCGCCGGGCAGATCGATGCGCCGGCCGGCGCGACCGTCGTCGATGCGCACGACCAGTACGTGATGCCGGGCGGCATCGATCCGCATACGCACATGGAACTGCCGTTCATGGGCACGACCGCGAGCGACGATTTCTACTCGGGTACGGCCGCCGGGCTCGCGGGCGGCACGACGAGCATCATCGACTTCGTGATCCCGAGCCCGAAGCAGCCGCTGATGGACGCGTTCCATGCCTGGCGCGGCTGGGCCGAGAAGGCGGCGGCCGACTACGGCTTCCACGTGGCCGTGACGTGGTGGGACGAGAGTGTGCACCGCGACATGGGCACGCTCGTGCGCGAACACGGCGTGTCGAGCTTCAAGCACTTCATGGCGTACAAGAACGCGATCATGGCCGACGACGAGGTGCTCGTGAACAGCTTCTCGCGTTCGCTCGAACTCGGCGCGTTGCCGACCGTGCATGCGGAGAACGGCGAGCTCGTGTTCCAGTTGCAGAAGGCGCTGCTCGCGCGCGGGATGACGGGGCCGGAGGCGCATCCGCTGTCGCGGCCGCCGGAGGTCGAGGGTGAGGCGGCGAATCGTGCGATCCGCATTGCGCAGGTGCTCGGCGTGCCGGTGTATATCGTGCATGTGTCCGCGAAGGACGCGGTCGATGCGATCACGAAGGCGCGCAGCGAAGGGCTGCGCGTGTTCGGCGAGGTGCTGCCGGGCCATCTGGTGATCGACGAGGCCGTCTATCGCGATCCGGACTGGACACGTGCG

***5. What is the starting position of the longest ORF in reading frame 3 in any of the sequences? The position should indicate the character number where the ORF begins.***

In [134]:
rfp = 3
orf_length_dict4, orf_longest_seq_dict4, orf_longest_file4 = orf("dna2.fasta", reading_frame_pos=rfp)

pprint(orf_longest_file4)

{'gi|142022655|gb|EQ086233.1|527 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'636-2456': 'ATGAACAGCGGGGCGAGCAAGCCGCCGGCCGTCACGGGGTCCATCACGAGGGACAGCAGCGGAATGCCGATGATCGCGAATCCACCACCGAACGCGCCGCGCATGAACGCGATCACGAACACGCCGGCAAACGCGATCAGGATCGTGGCCAGCGTCAATTGCAGGCCCATCGCAGCAGGGGTCGCCATCACGACCTCCATGCCGGTTCGAATCGCGGCGTGGCGGACAGCCACGGAGCGGGTCGCACGCGCGGCATCGCCGCACGATGGATCCGGGTTGAACGCGTTGCACCCATGCTGCTTCTCCAATGAGGTACCGGGGCGATGCGGTACACCAACGCACCGCAGGCCGCATGGGCCGCACAAGCATTTCAGCCCCGGTACAATCGACTTGACGAAAGCAGAATGCACCGCCGTCTATCTCAGTGCAATTAAAACATTGACCTCGGTGCAATATTCATTGTTATCGGTGCAATCCATGTCGAATTCCGAATACCTGCAGTTGGCCGACGCGATCGCCGCCCAAATTGCCGACGGCACGCTCAGGCCGGGCGACCGCCTGCCTCCGCAGCGTCATTTCGCCGACCAGCATGCGATCGCCGCATCGACGGCGGGACGGGTTTACGCGGAACTGTTACGGCGCGGCCTTGTGGTCGGCGAAGTCGGCCGAGGCACTTTCGTGTCGGGTGAGACGCGACGCGGGGCCGCTGCGCCGGGCGAGCCGCGCGGCGTTCGGATCGATTTCGAGTTCAACTACCCGACCGTCCCGGCCCAGACCGCGTTGATCACCAGAAGCCTGCGCGGATTGCACCGACCTGCGGAGCTCGACGCCGCGTTACGCGAGGCGACGAGTACCGGGACC

***6. What is the length of the longest ORF appearing in any sequence and in any forward reading frame?***

In [135]:
longest_orf_per_rfpos = {}
for rfpos in [1,2,3]:
    _length_dict, _orf_longest_per_seq, orf_longest_file_rfp = orf(file_name="dna2.fasta", reading_frame_pos=rfpos)
    inner_dict_rfp = next(iter(orf_longest_file_rfp.values()))
    longest_orf_per_rfpos[f"Reading frame {rfpos}"] = len(next(iter(inner_dict_rfp.values())))
    pprint(orf_longest_file_rfp)

print()
pprint(longest_orf_per_rfpos)
rf_with_longest_orf = max(longest_orf_per_rfpos, key=longest_orf_per_rfpos.get)
print(f"{rf_with_longest_orf} shows the longest ORF with length {longest_orf_per_rfpos[rf_with_longest_orf]}")

{'gi|142022655|gb|EQ086233.1|45 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'385-2778': 'ATGGAGAAACAGTCTCGCGTTACGCGCGACGGTCGCGGGAGAGTTCTATGCGGTCATCGCTGCCGCGGTCGCGATTGGACTGGTCATGACGTTCGTTCATTTCGACCCGATTCGAGCGCTCTACTGGAGCGCCGTCATCAATGGGATCACGGCAGTGCCCATCATGGTGGTGATGATGCTGATGGCGCAGAGCCGGCGCGTGATGGGCGAGTTCGCAATCAGAGGACCGCTTGCGTGGGGAGGGTGGCTCGCGACGCTCGCCATGGCGCTCGCGGCGGCCGGAATGCTGCTGCCGGGATGAGCCGGCAATCCGGATGGAGAATGCGCATGCCCGCGACGCACCGGCGACGCCTCGCCGGACGGCGGGCGTCGCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCATTCGCCGAGCGCTCCATCGACGACGGTGGCGGCCACGCCCCGGAATTCGACATGCCTGCATCCTCCGATACGGCGAACCGGCGGGCGTCATCAATCGCGCGCATCCAGCGCGGGCTGAAGCGCGGGCTCGGCCGGCGCTGCCGGTTCATGGCCGCCGTGGCGCGCGGCGGTGGAATGGCCGGGCCGGATCCTGAACCAGATCGCATACATCGCGGGCAGGAACACGAGCGTGAGGACCGTCCCGGCGAACGTGCCGCCGATCAGCGTGTACGCGAGCGTGCCCCAGAACACCGAATGCGTGAGCGGAATGAACGCGAGCACGGCCGCCATCGCGGTAAGAATCACCGGGCGCGCCCGCTGCACGGTCGCTTCGACGACCGCGTGGAACGGATCGAGTCCCGCGTGTTCGTTCTGGTGG

***7. What is the length of the longest forward ORF that appears in the sequence with the identifier  gi|142022655|gb|EQ086233.1|16?***

In [136]:
answer_dict = {}

for rfpos in [1,2,3]:
    _length_dict, orf_longest_per_seq, _orf_longest_file_rfp = orf(file_name="dna2.fasta", reading_frame_pos=rfpos)
    keys = list(orf_longest_per_seq.keys())

    given_key = [
        key
        for key in keys
        if "gi|142022655|gb|EQ086233.1|16" in key
        ]
    given_key = ''.join(given_key)

    given_seq = orf_longest_per_seq[given_key]
    pprint(given_seq)
    answer_dict[f"Reading frame {rfpos}"] = len(next(iter(given_seq.values())))

pprint(answer_dict)
longest_rf = max(answer_dict, key=answer_dict.get)
print()
print(f"The longest ORF for this specific sequence is in reading frame {longest_rf} with length {answer_dict[longest_rf]}")

{'1528-3036': 'ATGAATCACGCAGCGAATCCCGCCGATCCCGATCGCGCCGCGGCGCAGGGCGGCAGCCTGTACAACGACGATCTCGCGCCGACGACGCCGGCGCAGCGCACGTGGAAGTGGTATCACTTCGCGGCGCTGTGGGTCGGGATGGTGATGAACATCGCGTCGTACATGCTCGCGGCCGGGCTGATCCAGGAAGGCATGTCGCCGTGGCAGGCGGTGACGACGGTGCTGCTCGGCAACCTGATCGTGCTCGTGCCGATGCTGCTGATCGGCCATGCGGGCGCGAAGCACGGGATTCCGTACGCGGTGCTCGTGCGCGCGTCGTTCGGCACGCAGGGGGCGAAGCTGCCGGCGCTGCTGCGCGCGATCGTCGCGTGCGGCTGGTACGGGATCCAGACCTGGCTCGGCGGCAGCGCGATCTATACGCTGCTGAACATCCTGACCGGCAACGCGCTGCATGGCGCCGCGCTGCCGGTCATCGGCATCGGGTTCGGGCAGCTCGCATGCTTCCTCGTGTTCTGGGCGCTGCAGCTCTACTTCATCTGGCATGGCACCGATTCGATCCGCTGGCTCGAAAGCTGGTCGGCGCCGATCAAGGTCGTGATGTGCGTGGCGCTGGTGTGGTGGGCAACGTCGAAGGCGGGCGGCTTCGGCACGATGCTGTCGGCGCCGTCGCAGTTTGCCGCAGGCGGCAAGAAAGCCGGGCTGTTCTGGGCGACCTTCTGGCCGGGGCTGACCGCGATGGTCGGCTTCTGGGCGACGCTCGCGCTGAACATCCCCGACTTCACGCGCTTCGCGCATTCGCAGCGCGACCAGGTGATCGGCCAGTCGATCGGGCTGCCGTTGCCGATGGCGCTGCTGTCGGTGGTGTCGGTCGTCGTGACGTCGGCGACCGTCGTGATCTACGGCAACGCGATCTGGGATCCGATCGACCTGACGAGCCGGATGACGGGCATCGGCGTGGGCATCGCGCTCGTGATCCTCACGCTCG

***8. Find the most frequently occurring repeat of length 6 in all sequences. How many times does it occur in all?***

In [137]:
repeats6_dict = repeats_inefficient(file_name="dna2.fasta", length=6)
most_repeat6 = max(repeats6_dict, key=repeats6_dict.get)
print(f"The msot frequent repeat of length 6 in the file is {most_repeat6} with {repeats6_dict[most_repeat6]} repetitions.")

The msot frequent repeat of length 6 in the file is GCGCGC with 153 repetitions.


***9. Find all repeats of length 12 in the input file. Let's use Max to specify the number of copies of the most frequent repeat of length 12.  How many different 12-base sequences occur Max times?***

In [138]:
repeats12_dict = repeats_inefficient(file_name="dna2.fasta", length=12)
most_frequent12 = max(repeats12_dict, key=repeats12_dict.get)
frequency_most12 = repeats12_dict[most_frequent12]
most_freq_12repeats = {
    repeat: freq
    for repeat, freq in repeats12_dict.items()
    if freq == frequency_most12
}
pprint(most_freq_12repeats)
print(f"The number of repeats of length 12 that are equally most frequent are {len(most_freq_12repeats)}")

{'ATTCGCCATTCG': 10, 'CATTCGCCATTC': 10, 'TCGCCATTCGCC': 10, 'TTCGCCATTCGC': 10}
The number of repeats of length 12 that are equally most frequent are 4


***10. Which one of the following repeats of length 7 has a maximum number of occurrences?***

In [139]:
repeats7_dict = repeats_inefficient(file_name="dna2.fasta", length=7)
most_frequent7 = max(repeats7_dict, key=repeats7_dict.get)
frequency_most7 = repeats7_dict[most_frequent7]
most_freq_7repeats = {
    repeat: freq
    for repeat, freq in repeats7_dict.items()
    if freq == frequency_most7
}
pprint(most_freq_7repeats)

{'CGCGCCG': 63}


# *Everything below is a testing bay for the questions above.*

---

In [140]:
r = {'r':'ready','d':'dog', 'z':'hippopotamus'}
print(r)
length_dict = {seq_id:len(seq) for seq_id,seq in r.items()}
print(length_dict)
print(max(length_dict.values()))
print(list(length_dict.values()))

{'r': 'ready', 'd': 'dog', 'z': 'hippopotamus'}
{'r': 5, 'd': 3, 'z': 12}
12
[5, 3, 12]


In [141]:
sequence1 = "ATGATGTAAATGATGTGAGATGTGACATGATGTAG"
print(sequence1)

rf = 1

i = rf - 1
if rf > 1:
    codons = [sequence1[:i]]
else:
    codons = []

while i+3 <= len(sequence1):
    codons.append(sequence1[i:i+3])
    i += 3

if i != len(sequence1):
    codons.append(sequence1[i:])
print(codons)

rfs = seq_rf(sequence1, rf)
print(rfs)

start_codon_indices1 = [
    index
    for index, value in enumerate(rfs)
    if value == 'ATG'
    ]
print(f"Start codon positions: {start_codon_indices1}")

stop_codon_indices1 = [
    index
    for index, value in enumerate(rfs)
    if value in ('TAG', 'TAA', 'TGA')
]
print(f"Stop codons positions: {stop_codon_indices1}")

prev_stop_codon = 0
open_rfs = {}
pos_list = [1, -1, 0]

for stop_codon_index in stop_codon_indices1:
    start_codon_indices_before1 = [
        start_codon_index
        for start_codon_index in start_codon_indices1
        if (prev_stop_codon <= start_codon_index < stop_codon_index)
    ]

    for scib in start_codon_indices_before1:
        open_rfs[f"{scib * 3 + pos_list[rf - 1]}-{stop_codon_index * 3 + pos_list[rf - 1] + 2}"] = "".join(rfs[scib:stop_codon_index + 1])
        # print(f"{scib}-{stop_codon_index}: {"".join(rfs[scib:stop_codon_index + 1])}")

    # print(open_rfs)    
    prev_stop_codon = stop_codon_index
print()
pprint(open_rfs)
seq_orfs =  {}
seq_orfs[sequence1] = open_rfs
pprint(seq_orfs)

ATGATGTAAATGATGTGAGATGTGACATGATGTAG
['ATG', 'ATG', 'TAA', 'ATG', 'ATG', 'TGA', 'GAT', 'GTG', 'ACA', 'TGA', 'TGT', 'AG']
['ATG', 'ATG', 'TAA', 'ATG', 'ATG', 'TGA', 'GAT', 'GTG', 'ACA', 'TGA', 'TGT', 'AG']
Start codon positions: [0, 1, 3, 4]
Stop codons positions: [2, 5, 9]

{'1-9': 'ATGATGTAA', '10-18': 'ATGATGTGA', '13-18': 'ATGTGA', '4-9': 'ATGTAA'}
{'ATGATGTAAATGATGTGAGATGTGACATGATGTAG': {'1-9': 'ATGATGTAA',
                                         '10-18': 'ATGATGTGA',
                                         '13-18': 'ATGTGA',
                                         '4-9': 'ATGTAA'}}


In [142]:
seq_len_orfs = {
    seq_id:{
        pos:max_len_orf
        for pos, max_len_orf in orfs_dict.items()
        if len(max_len_orf) == len(max(orfs_dict.values(), key=len))}
    for seq_id, orfs_dict in seq_orfs.items()
}
print(seq_len_orfs)

{'ATGATGTAAATGATGTGAGATGTGACATGATGTAG': {'1-9': 'ATGATGTAA', '10-18': 'ATGATGTGA'}}


In [143]:
# testing bay for q3

ex_seq_dict = {'gi|142022655|gb|EQ086233.1|101 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'1633-2271': 'ATGGTCGCGGCGGCAGGCAGCGACGAACCGCTCCTGGAGCAACATCTTGAACTCGATGTCGGATTCCTGGCTGCCCATGAAGCTCACGCCGAAATCGGCTTCGCCGCTGATGACGGCGCCCAGCACCTCGTTCGCGCTCGCGTCCAGCAGCTTGACCCGGATGCGCGGAAAGCGCTGATGATAGCGCGCGATGATGGCCGGCAGAAAGTAGTAGGCGACCGAGGGCACGCACGCGATGGTCACATGGCCCAGGCGGCTCGACGACACGTCGCGAATGCCGAGCAGCGCCGCATCGAGATCGTCGAGCAGCTGTTCGGCGCTCTGGGCGAACACGCGGCCGACCGTGGTGAGCGCGACGCGACGCGTGGTGCGCTCGAACAGGCGCACGCCGAGCGCTTCCTCGAGCTTGTCGATCCGGCGACTCAACGCGGGCTGGGAAATGCTGACCGATTCCGCGGCCTTGCGGAAACTGCCCGTTTCCACGACCGCGCGAAACGCCTGCAAGTCGTTCAAGTCGAAGTTGATCCCCACGGGCGCGTCTCCCCATCTCAGATGGGGCGTATTTTGCATGATTTCGCCGGGCGGCCGCATCGGCGCGGCACGCATTCGCGCCACCCTCGATCGCAACCGCGTGCGTGA'},
 'gi|142022655|gb|EQ086233.1|158 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'700-828': 'ATGCCGTCCAGCTTGCCGTGGTCGTGGCGGCGCTGTTTTTCGGCGCGTTCATGCATCCTGCCGTCAACACAAGCGAGGAAGCGGTCGCGAAGGACGCCATCAAATCCAGTGACGTCCGAGGGATTCTGA'},
 'gi|142022655|gb|EQ086233.1|160 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'136-444': 'ATGGGGGCGACGGTGTATTTCCGCCAGAAGATTTCGCCGCGGGAGCTCGCGGTGCGTACGTGCATGTTCAAACGCACGGTGCGCGCATGGCAGTGGCAGACTGATCAACGCAGCTGGAAGCATCCGAAGCGCGCGGGCACGCGTGTCCTCGACGCGTGGCCTCACATGCTGTCGGGTCGGTTCAAGACCGAAAGCCACCGACCGACGCGCGAGCAATGCGCTACGCGGATCGCGTTCGACACGAGCCGCGCGCGAGGCAAGGCCGACGTATTCGATCTTCCAGAGGAAGCCTATTGGCTCGAGTCGTAG'},
 'gi|142022655|gb|EQ086233.1|210 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'478-888': 'ATGCGTAGAAGCCGATCAGCATCTCGAGATTGCGTTGCGCGTGCTGCTCGAGCGGCGCGAATTCCGCCGATCGAAACAACGGGGCCTGCGCGCGCAGATGCTCGAGCGCCTGCCTGCGGCCGGCGCGGAAGGCTGCCGACGGTGCATCGGGATCCTGCGCCCACTCGCGCGCGATCGCGCGGTCGTATTCGCGGAGCCGGTCGGGTGCCGCCGCGAGGATCGCGAGATCGGCGTCGAGGAAAATTTGCGCGGCGCGCTGCAATTCCGCATCGTCGGCGAACCCGTCAGGCAGCCGGTGCGACTTCGTCGCCAGCACCAGGTCGCGGGCAACGGACACGTGCGATGCATGCGCGTGCAACCAGGCTGCGTCGCAATGTTCGTGCGCGACCTGCGCGAGCCATTGCGCGCTGA'},
 'gi|142022655|gb|EQ086233.1|221 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'1246-1509': 'ATGGCCCGCGCTGCGGGTCGTCGCGGCCAAAGGCTCCGACCTGCGCCTGACCTCGCGGGATGCCACGCGACGCATGATCACCGCCTGATCCGCGTCCGCGCGTGCTCACGACGAATCTGCTCGACGCGCACGCCTATTCGGCCGCCGACATTGCTGCGCTTTAACGCCGCCTCCAAGGTACGAAGTCGCTGGCATCGTTTCTCGATCTGTTTAACCAGCGAGTTGCACGTCCGTTGCCCTTTCGACAAGGCTGACTTGCCGTGA'},
 'gi|142022655|gb|EQ086233.1|229 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'1960-2685': 'ATGACTGCGCCGACTGCACCGCCGGCCGACACGGCCAGGCGTCCGGCGACGGTGACGGACGCGATTCGCGCAGGCGTTGGGATCGGGCCTGCTCGAATGGCTGTTATTTCGCTATCCGAACGGATATGCGGCGCGATGGCATTCGCACTCCTCGGCTACCTGGCTTGCGACTTGCCGACCCGGCTGCACGGTGCTCCTTCCAGCTGGATATGGAACGGCATCGGCGCAATGGTTTGCGGCATCGCCGGCTATTCGATGCAAGGCGGCCTGTCCGGCGGGTGCGCGCGCAGGGCAATGCTCCTGTGCACGGGGGCCATCGGCGCGGGATTGACGCTCGATGCGATGCGGTCCCCCGTCGATGCCATTCTCGACATCTGCGGCGGAACGCTCGATGCCCGCGCCATGTGGAACACACTCGTGCTGCATCTCCAATGGTTCCCGATGTCGATGCTGGCCATGCTGGCGCTGCTGATCTTGCGCGAGGCCGACCGCAAGGACCGTCGCCGACGCGGCGCGGCAAGCATGCTCGTTGTGGTGCCGGTGCGCGTCGCGCTCGAATTCGCGGGGATGCAGCTCGTCATGGCGCTCGGCATGACCGCGATCAGGGCATCGGCGTTCGCTGCCGGGCTGCGTTGGGATACGAGCGGCGTGGCAGTGTCGATGCTCGCCAGCATGCTGGCGTTCGACGCATTGAGCGATCGGGCGGCACGCGCAGGGGGGCGTTGA'},
 'gi|142022655|gb|EQ086233.1|237 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'1027-1200': 'ATGGCCGCGCGTCTCGCCCGGCGCGCGGTGGGCGCGCCGGACTTTGGCGATGGGCGTTACGACGGCACGCCTTTCAGCGAGCCGTCGCGATGCGGTGCGAATGACGCCGGTTCCGGCGCCGCGGCATCGGCGGCCGGCACGAAGAACAGCGAACCGGTAACCGCGCGGCTGTAG',
                                                                                                                              '2479-2652': 'ATGCTCACGTCTTCTAGCAGACGGGGAATTGGGACGGGACTGCGTAGTCAAGTAAGCAGTCCCGGTCTCTCGAATTCAATGCGACATTTTCGGGGGCGGCCGAAAAGCGCAACTTTCACCGGCTACCCTTTGATTGACCAAACCTTGGCTGCAATTTCGGCCATCGCAGCTTGA'}}

def max_length_orf_total(seq_dict):
    '''Return longest orf in the entire file.
        Args:
            seq_dict (dict) - nested dictionary of sequences with their longest orf(s) and their respective positions.
        
        Returns:
            dict
    '''
    longest_len_so_far = 0
    longest_orf_so_far = {}

    for seq_id, orf_dict in seq_dict.items():
        orf_curr = next(iter(orf_dict.values()), None)
        if orf_curr is None:
            continue

        len_orf_curr = len(orf_curr)

        if len_orf_curr > longest_len_so_far:
            longest_len_so_far = len_orf_curr
            longest_orf_so_far = {seq_id: orf_dict}
        elif len_orf_curr == longest_len_so_far:
            longest_orf_so_far.update({seq_id: orf_dict})

    return longest_orf_so_far
        
longest_orf_infile = max_length_orf_total(ex_seq_dict)
print(longest_orf_infile)

for seq_id, orfs in longest_orf_infile.items():
    print(seq_id.split()[0], {pos: len(seq) for pos, seq in orfs.items()})

{'gi|142022655|gb|EQ086233.1|229 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'1960-2685': 'ATGACTGCGCCGACTGCACCGCCGGCCGACACGGCCAGGCGTCCGGCGACGGTGACGGACGCGATTCGCGCAGGCGTTGGGATCGGGCCTGCTCGAATGGCTGTTATTTCGCTATCCGAACGGATATGCGGCGCGATGGCATTCGCACTCCTCGGCTACCTGGCTTGCGACTTGCCGACCCGGCTGCACGGTGCTCCTTCCAGCTGGATATGGAACGGCATCGGCGCAATGGTTTGCGGCATCGCCGGCTATTCGATGCAAGGCGGCCTGTCCGGCGGGTGCGCGCGCAGGGCAATGCTCCTGTGCACGGGGGCCATCGGCGCGGGATTGACGCTCGATGCGATGCGGTCCCCCGTCGATGCCATTCTCGACATCTGCGGCGGAACGCTCGATGCCCGCGCCATGTGGAACACACTCGTGCTGCATCTCCAATGGTTCCCGATGTCGATGCTGGCCATGCTGGCGCTGCTGATCTTGCGCGAGGCCGACCGCAAGGACCGTCGCCGACGCGGCGCGGCAAGCATGCTCGTTGTGGTGCCGGTGCGCGTCGCGCTCGAATTCGCGGGGATGCAGCTCGTCATGGCGCTCGGCATGACCGCGATCAGGGCATCGGCGTTCGCTGCCGGGCTGCGTTGGGATACGAGCGGCGTGGCAGTGTCGATGCTCGCCAGCATGCTGGCGTTCGACGCATTGAGCGATCGGGCGGCACGCGCAGGGGGGCGTTGA'}}
gi|142022655|gb|EQ086233.1|229 {'1960-2685': 726}


In [144]:
# testing bay for question 4

string1 = 'readforme'
print(string1[0:3])


length = 3
print(len(string1))

print(string1[6: 6 + length])
counter = 0
while counter + length <= len(string1):
    print(string1[counter:counter + length])
    print(counter + length)
    counter +=1

rea
9
rme
rea
3
ead
4
adf
5
dfo
6
for
7
orm
8
rme
9


In [145]:
help(dict)

Help on class dict in module builtins:

class dict(object)
 |  dict() -> new empty dictionary
 |  dict(mapping) -> new dictionary initialized from a mapping object's
 |      (key, value) pairs
 |  dict(iterable) -> new dictionary initialized as if via:
 |      d = {}
 |      for k, v in iterable:
 |          d[k] = v
 |  dict(**kwargs) -> new dictionary initialized with the name=value pairs
 |      in the keyword argument list.  For example:  dict(one=1, two=2)
 |
 |  Methods defined here:
 |
 |  __contains__(self, key, /)
 |      True if the dictionary has the specified key, else False.
 |
 |  __delitem__(self, key, /)
 |      Delete self[key].
 |
 |  __eq__(self, value, /)
 |      Return self==value.
 |
 |  __ge__(self, value, /)
 |      Return self>=value.
 |
 |  __getattribute__(self, name, /)
 |      Return getattr(self, name).
 |
 |  __getitem__(self, key, /)
 |      Return self[key].
 |
 |  __gt__(self, value, /)
 |      Return self>value.
 |
 |  __init__(self, /, *args, **kwarg

In [146]:
help(max)

Help on built-in function max in module builtins:

max(...)
    max(iterable, *[, default=obj, key=func]) -> value
    max(arg1, arg2, *args, *[, key=func]) -> value

    With a single iterable argument, return its biggest item. The
    default keyword-only argument specifies an object to return if
    the provided iterable is empty.
    With two or more positional arguments, return the largest argument.

